# Always-on Kafka streams: start, feed, watch, stop

`--available-now` streams stop by themselves. Real streaming jobs don't: they run until you
kill them. Here we start jobs 31, 32 and 33 always-on through Kyuubi (1 core each), feed
them with the producer for a couple of minutes, watch Postgres and Kafka fill up, then
kill all three. The worker has 4 cores, so don't run other jobs meanwhile.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
import time, lab_runner
assert lab_runner.kafka_reachable(), "Kafka isn't running - make kafka-install"
STREAMS = ["kafka-to-delta-pg", "kafka-enrich-publish", "kafka-windowed-metrics"]
ids = {n: lab.submit_no_wait(n, RID, extra_args=lab_runner.case(n).get("always_on") or [],
                             conf={"spark.cores.max": "1"}, batch_name=f"nb-live-{n}")
       for n in STREAMS}
ids

In [ ]:
for _ in range(60):
    states = {n: lab.state(b) for n, b in ids.items()}
    if all(s == "RUNNING" for s in states.values()) or any(s in ("ERROR", "FINISHED") for s in states.values()):
        break
    time.sleep(5)
states

Feed them for about two minutes (a few events per second).

In [ ]:
# a new run id per round, so every round brings new orders (RID-0, RID-1, ...)
for i in range(8):
    lab.produce(orders=10, run_id=f"{RID}-{i}")
    time.sleep(15)
    n = lab.pg("SELECT count(*) n FROM kafka_order_status WHERE run_id LIKE %s", (RID + "-%",))["n"][0]
    print(f"round {i}: {n} orders tracked in Postgres so far")

In [ ]:
enr = lab.topic('orders.enriched', last=50)
enr[enr["run_id"].str.startswith(RID)].tail(10) if "run_id" in enr else enr

In [ ]:
lab.pg("SELECT * FROM order_metrics_live ORDER BY window_start DESC LIMIT 10")

## Stop everything
Always do this, or the streams keep the worker's cores.

In [ ]:
for n, b in ids.items():
    lab.kill(b)
time.sleep(5)
{n: lab.state(b) for n, b in ids.items()}